In [ ]:
#pip install transformers datasets sacrebleu sentencepiece accelerate

In [53]:
import pandas as pd
import sentencepiece as spm
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

In [54]:
tf.random.set_seed(42)
np.random.seed(42)

In [55]:
train_df = pd.read_csv("/home/sibel/Langue-wu/Data/Corpus_aligné/train.csv")
dev_df   = pd.read_csv("/home/sibel/Langue-wu/Data/Corpus_aligné/dev.csv")
test_df  = pd.read_csv("/home/sibel/Langue-wu/Data/Corpus_aligné/test.csv")

train_df.head()

,mandarin,wu
0,更不会对美接触过得人有感觉！,更加伐会对没接触过呃拧有感觉！
1,我没找啊！我就说没几个长的好看的,吾没寻啊！吾就讲没记几个长勒好看呃
2,哈哈哈 祝您开心！,哈哈 祝侬开心！
3,我哭了 所以才问你,吾哭了 所以才再问侬
4,给他讲清道理 不厌烦的讲,帮伊讲清道理 伐厌烦呃讲


In [56]:
with open("sp_corpus.txt", "w", encoding="utf-8") as f:
    for t in train_df["wu"]:
        f.write(t + "\n")
    for t in train_df["mandarin"]:
        f.write(t + "\n")

In [57]:
spm.SentencePieceTrainer.Train(
    "--input=sp_corpus.txt --model_prefix=sp --vocab_size=4000 --character_coverage=1.0"
)

sp = spm.SentencePieceProcessor()
sp.load("sp.model")

PAD_ID = sp.pad_id()
BOS_ID = sp.bos_id()
EOS_ID = sp.eos_id()
VOCAB_SIZE = sp.get_piece_size()

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=sp_corpus.txt --model_prefix=sp --vocab_size=4000 --character_coverage=1.0
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: sp_corpus.txt
  input_format: 
  model_prefix: sp
  model_type: UNIGRAM
  vocab_size: 4000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  

In [58]:
print(f"Vocab size: {VOCAB_SIZE}")
print(f"PAD_ID: {PAD_ID}, BOS_ID: {BOS_ID}, EOS_ID: {EOS_ID}")

Vocab size: 4000
PAD_ID: -1, BOS_ID: 1, EOS_ID: 2


In [39]:
def encode(text):
    return [BOS_ID] + sp.encode_as_ids(text) + [EOS_ID]

In [ ]:
def decode(ids):
    return sp.decode_ids(ids)

In [40]:
def make_dataset(df, batch_size=32):
    src = [encode(x) for x in df["wu"]]
    tgt = [encode(x) for x in df["mandarin"]]
    
    src = tf.keras.preprocessing.sequence.pad_sequences(src, padding="post", value=PAD_ID)
    tgt = tf.keras.preprocessing.sequence.pad_sequences(tgt, padding="post", value=PAD_ID)
    
    decoder_in = tgt[:, :-1]
    decoder_out = tgt[:, 1:]
    
    dataset = tf.data.Dataset.from_tensor_slices((
        {"encoder_input": src, "decoder_input": decoder_in},
        decoder_out
    ))
    
    return dataset.shuffle(1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [41]:
train_ds = make_dataset(train_df, batch_size=32)
dev_ds = make_dataset(dev_df, batch_size=32)